# Notebook 02: Baseline Training & Optimization Experiments with YOLOv8n

**Objectives:**
1. Load pretrained `yolov8n.pt` (COCO weights) and explain the architecture.
2. Train a baseline detector at `imgsz=640` for 50 epochs with Early Stopping.
3. Dissect the 3 loss functions: Box Loss (CIoU), Class Loss (VFL/BCE), and DFL (Distribution Focal Loss).
4. Conduct the **Step 6 Improvement Experiment**: train with high-resolution input (`imgsz=960`) to evaluate small-object recall improvements.
5. Compare baseline vs. experiment metrics and training time.

### Why YOLOv8n (Nano)?
- **Parameters:** ~3.2 Million (extremely lightweight for mobile & edge devices).
- **Anchor-Free Architecture:** Eliminates manual anchor box tuning; directly predicts distance from grid centers to the 4 bounding box edges.
- **C2f Modules:** Cross-Stage Partial bottleneck blocks with multi-branch skip connections for rich gradient flow.
- **Decoupled Head:** Separates the classification branch from the regression branch, avoiding gradient competition.

In [ ]:
# Cell 1: Environment & Ultralytics Import
import os
import torch
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Baseline Training (imgsz=640)
# Estimated time on free Colab T4 GPU: ~40-50 seconds per epoch (~25 minutes for 35-50 epochs)
# -------------------------------------------------------------------------
# Key Hyperparameters:
# - data: path to data.yaml
# - imgsz: 640 (standard input resolution)
# - epochs: 50 (with patience=10 for early stopping)
# - batch: 16 (fits comfortably in T4 15GB VRAM)
# - seed: 42 (reproducibility)

model_baseline = YOLO('yolov8n.pt') # loads COCO pretrained weights

print("Starting baseline training...")
results_baseline = model_baseline.train(
    data='../data/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    seed=42,
    project='../runs/train',
    name='baseline_yolov8n',
    save=True,
    verbose=True
)

### Understanding the Loss Functions in YOLOv8
1. **Box Loss (CIoU):** Complete Intersection over Union loss considers three geometric factors: overlap area, central point distance, and aspect ratio consistency:
   $$\mathcal{L}_{\text{CIoU}} = 1 - \text{IoU} + \frac{\rho^2(b, b^{\text{gt}})}{c^2} + \alpha v$$
2. **Class Loss (VFL / BCE):** Binary Cross Entropy measuring accuracy of predicted class logits.
3. **DFL Loss (Distribution Focal Loss):** Represents continuous bounding box coordinates as a discrete probability distribution over neighboring grid points. It allows the model to learn spatial uncertainty around blurry or occluded edges.

In [ ]:
# Cell 3: Step 6 Improvement Experiment (High Resolution: imgsz=960)
# -------------------------------------------------------------------------
# Hypothesis: Workers distant from site cameras appear small (< 32px).
# Scaling input resolution to 960x960 preserves fine features (helmet straps, head contours)
# before aggressive spatial downsampling.

model_exp = YOLO('yolov8n.pt')

print("Starting high-resolution experiment (imgsz=960)...")
results_exp = model_exp.train(
    data='../data/data.yaml',
    epochs=50,
    imgsz=960,
    batch=8,     # reduce batch size to fit higher resolution in VRAM
    patience=10,
    seed=42,
    project='../runs/train',
    name='experiment_imgsz960',
    save=True
)